# Transition Matrix with 2D Grid: Serial Permanent Income Growth

**MC vs TM comparison for a 5-state Markov model with varying PermGroFac**

This notebook tackles the key new challenge that the previous prototypes avoided:
**PermGroFac $\neq$ 1.0**.  When permanent income growth differs across Markov
states, the distribution over permanent income levels $p$ is non-degenerate.
We must track the joint distribution over $(m, p, j)$ using a **2D grid**
$(m \times p)$ for each of the $J=5$ Markov states.

| State | PermGroFac | Interpretation |
|-------|-----------|----------------|
| 0     | 0.97      | Deep contraction |
| 1     | 0.99      | Mild contraction |
| 2     | 1.01      | Normal growth |
| 3     | 1.03      | Expansion |
| 4     | 1.05      | Boom |

---

In [ ]:
import time
from copy import copy

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse.linalg as sp_linalg

from HARK.ConsumptionSaving.ConsMarkovModel import (
    MarkovConsumerType,
    init_indshk_markov,
)
from HARK.ConsumptionSaving.ConsMarkovModel import make_markov_solution_terminal
from HARK.utilities import make_grid_exp_mult, jump_to_grid_2D

## 1. Model Setup

5 Markov states with persistence 0.5 and equal off-diagonal transitions.
All states share the same Rfree, LivPrb, and income process.
Only PermGroFac varies.

In [ ]:
J = 5
Persistence = 0.5
PermGroFac_vals = np.array([0.97, 0.99, 1.01, 1.03, 1.05])
state_names = [f"PermGroFac={g:.2f}" for g in PermGroFac_vals]

# Row-stochastic Markov matrix: MrkvArray[i,j] = P(go to j | in i)
MrkvArray = np.zeros((J, J))
for j_from in range(J):
    for j_to in range(J):
        if j_from == j_to:
            MrkvArray[j_from, j_to] = Persistence
        else:
            MrkvArray[j_from, j_to] = (1.0 - Persistence) / (J - 1)

print("Markov transition matrix (row-stochastic):")
print(np.array2string(MrkvArray, precision=3))
print(f"Row sums: {MrkvArray.sum(axis=1)}")

eigvals, eigvecs = np.linalg.eig(MrkvArray.T)
idx = np.argmin(np.abs(eigvals - 1.0))
markov_stationary = eigvecs[:, idx].real
markov_stationary = markov_stationary / markov_stationary.sum()
print(f"Stationary distribution: {markov_stationary}")

## 2. Solve the Model

In [ ]:
# Build income distribution: standard lognormal + unemployment, same for all states
from HARK.Calibration.Income.IncomeProcesses import (
    construct_lognormal_income_process_unemployment,
)

base_dstn_list = construct_lognormal_income_process_unemployment(
    T_cycle=1,
    PermShkStd=[0.1],
    PermShkCount=5,
    TranShkStd=[0.1],
    TranShkCount=5,
    T_retire=0,
    UnempPrb=0.05,
    IncUnemp=0.3,
    UnempPrbRet=None,
    IncUnempRet=None,
    RNG=np.random.default_rng(0),
)
base_dstn = base_dstn_list[0]

# Replicate for J states (same income process in all states)
IncShkDstn_J = [base_dstn] * J

print(f"Income shocks: {len(base_dstn.atoms[0])} shock points")
print(
    f"Perm shock range: [{base_dstn.atoms[0].min():.3f}, {base_dstn.atoms[0].max():.3f}]"
)
print(
    f"Tran shock range: [{base_dstn.atoms[1].min():.3f}, {base_dstn.atoms[1].max():.3f}]"
)

In [ ]:
# Set up the Markov agent
params = copy(init_indshk_markov)
params["cycles"] = 0
params["AgentCount"] = 50000
params["T_sim"] = 1200
params["Rfree"] = [np.array(J * [1.03])]
params["LivPrb"] = [np.array(J * [0.98])]
params["PermGroFac"] = [PermGroFac_vals]
params["MrkvPrbsInit"] = markov_stationary
params["Mrkv_p11"] = [0.5]  # placeholder, overridden below
params["Mrkv_p22"] = [0.5]
params["global_markov"] = False

agent = MarkovConsumerType(**params)
agent.assign_parameters(MrkvArray=[MrkvArray])
agent.IncShkDstn = [IncShkDstn_J]
agent.solution_terminal = make_markov_solution_terminal(agent.CRRA, agent.MrkvArray)

agent.solve()
print(f"Solved: {len(agent.solution[0].cFunc)} consumption functions")
print(f"PermGroFac used: {agent.PermGroFac[0]}")

In [ ]:
m_plot = np.linspace(0.01, 10, 200)

plt.figure(figsize=(12, 7))
cmap = plt.cm.coolwarm
for j in range(J):
    c_vals = agent.solution[0].cFunc[j](m_plot)
    color = cmap(j / (J - 1))
    plt.plot(m_plot, c_vals, label=state_names[j], linewidth=2, color=color)
plt.plot(m_plot, m_plot, ":", color="gray", alpha=0.4, label="45° line")
plt.xlabel("Market resources $m$", fontsize=12)
plt.ylabel("Consumption $c$", fontsize=12)
plt.title("Consumption Functions by Permanent Growth State", fontsize=13)
plt.legend(fontsize=10)
plt.xlim([0, 10])
plt.ylim([0, 5])
plt.show()

## 3. Monte Carlo Simulation

In [ ]:
agent.track_vars = ["aNrm", "mNrm", "pLvl", "Mrkv"]
agent.initialize_sim()
agent.simulate()

# Aggregate in LEVEL terms: C = sum(c_i * p_i) / N, A = sum(a_i * p_i) / N
mc_cNrm = agent.state_now["mNrm"] - agent.state_now["aNrm"]
MC_C = np.mean(mc_cNrm * agent.state_now["pLvl"])
MC_A = np.mean(agent.state_now["aNrm"] * agent.state_now["pLvl"])

MC_C_nrm = np.mean(mc_cNrm)
MC_A_nrm = np.mean(agent.state_now["aNrm"])

print(f"MC Aggregate Consumption (level) = {MC_C:.6f}")
print(f"MC Aggregate Assets      (level) = {MC_A:.6f}")
print(f"MC Aggregate Consumption (norm)  = {MC_C_nrm:.6f}")
print(f"MC Aggregate Assets      (norm)  = {MC_A_nrm:.6f}")
print()
for j in range(J):
    frac = np.mean(agent.shocks["Mrkv"] == j)
    print(
        f"{state_names[j]:20s}: MC frac = {frac:.4f}, stat = {markov_stationary[j]:.4f}"
    )

In [ ]:
burn_in = 400
mc_aLvls = np.array(
    [
        np.mean(agent.history["aNrm"][t] * agent.history["pLvl"][t])
        for t in range(agent.T_sim)
    ]
)

## 4. Transition Matrix with 2D Grid

The state is now $(m, p, j)$ where $m$ is normalized market resources,
$p$ is the permanent income level, and $j$ is the Markov state.

The grid has $M \times P$ points for each Markov state, giving
$M \times P \times J$ total states.  We use `jump_to_grid_2D` to
distribute probability mass in the $(m, p)$ plane.

**Indexing:** State $(m_i, p_k, j)$ maps to index `j * (M * P) + i * P + k`.

In [ ]:
mMin, mMax, mCount, mFac = 0.001, 30, 80, 3
pMin, pMax, pCount = 0.1, 5.0, 25

dist_mGrid = make_grid_exp_mult(ming=mMin, maxg=mMax, ng=mCount, timestonest=mFac)
dist_pGrid = make_grid_exp_mult(ming=pMin, maxg=pMax, ng=pCount, timestonest=1)

M = len(dist_mGrid)
P = len(dist_pGrid)
MP = M * P
N_states = MP * J

print(f"m-grid: {M} points from {dist_mGrid[0]:.4f} to {dist_mGrid[-1]:.1f}")
print(f"p-grid: {P} points from {dist_pGrid[0]:.2f} to {dist_pGrid[-1]:.2f}")
print(f"Per-state grid: {M} × {P} = {MP}")
print(f"Total TM states: {N_states} ({J} states × {MP})")
print(f"TM memory: {N_states**2 * 8 / 1e6:.1f} MB")

In [ ]:
start = time.time()

MrkvArr = agent.MrkvArray[0]
Rfree_arr = agent.Rfree[0]
LivPrb_arr = agent.LivPrb[0]
PermGroFac_arr = agent.PermGroFac[0]
IncShkDstn_list = agent.IncShkDstn[0]

# Policy on m-grid (consumption function depends on m, not p)
cPol = []
aPol = []
for j in range(J):
    c_j = agent.solution[0].cFunc[j](dist_mGrid)
    a_j = np.maximum(dist_mGrid - c_j, 0.0)
    cPol.append(c_j)
    aPol.append(a_j)

# Newborn distribution: a=0, p=1.0, Markov state from stationary
MrkvPrbsInit = markov_stationary
NewBornDist = np.zeros(N_states)
for jp in range(J):
    shk_dstn = IncShkDstn_list[jp]
    tran_shks = shk_dstn.atoms[1]
    perm_shks = shk_dstn.atoms[0]
    # Newborn: m = tran_shk, p = 1.0 (for all shock realizations)
    newborn_2d = jump_to_grid_2D(
        tran_shks, np.ones_like(tran_shks), shk_dstn.pmv, dist_mGrid, dist_pGrid
    )
    NewBornDist[jp * MP : (jp + 1) * MP] = MrkvPrbsInit[jp] * newborn_2d

# Build transition matrix
TranMatrix = np.zeros((N_states, N_states))

for j in range(J):
    LivPrb_j = LivPrb_arr[j]

    for jp in range(J):
        markov_prob = MrkvArr[j, jp]  # row-stochastic
        if markov_prob < 1e-15:
            continue

        Rfree_jp = Rfree_arr[jp]
        PermGroFac_jp = PermGroFac_arr[jp]
        shk_dstn = IncShkDstn_list[jp]
        shk_prbs = shk_dstn.pmv
        perm_shks = shk_dstn.atoms[0]
        tran_shks = shk_dstn.atoms[1]

        for i in range(M):
            bNext_i = Rfree_jp * aPol[j][i]
            mNext_shks = bNext_i / (perm_shks * PermGroFac_jp) + tran_shks

            for k in range(P):
                pNext_shks = dist_pGrid[k] * perm_shks * PermGroFac_jp

                lottery_2d = jump_to_grid_2D(
                    mNext_shks, pNext_shks, shk_prbs, dist_mGrid, dist_pGrid
                )

                src_idx = j * MP + i * P + k
                TranMatrix[jp * MP : (jp + 1) * MP, src_idx] += (
                    markov_prob * LivPrb_j * lottery_2d
                )

    # Death/rebirth
    LivPrb_j = LivPrb_arr[j]
    for i in range(M):
        for k in range(P):
            src_idx = j * MP + i * P + k
            TranMatrix[:, src_idx] += (1.0 - LivPrb_j) * NewBornDist

elapsed = time.time() - start
print(f"Transition matrix built in {elapsed:.1f} seconds")
print(f"Shape: {TranMatrix.shape}")
col_sums = TranMatrix.sum(axis=0)
print(f"Column sums: min={col_sums.min():.8f}, max={col_sums.max():.8f}")

## 5. Ergodic Distribution

In [ ]:
start = time.time()
eigenvalues, eigenvectors = sp_linalg.eigs(
    TranMatrix, k=1, which="LM", v0=np.ones(N_states)
)
ergodic_dist = eigenvectors[:, 0].real
ergodic_dist = ergodic_dist / ergodic_dist.sum()

elapsed = time.time() - start
print(f"Ergodic distribution computed in {elapsed:.2f} seconds")
print()
for j in range(J):
    mass_j = ergodic_dist[j * MP : (j + 1) * MP].sum()
    print(
        f"{state_names[j]:20s}: TM mass = {mass_j:.4f}, stat = {markov_stationary[j]:.4f}"
    )

In [ ]:
# Compute aggregates in LEVEL terms: weight by p
TM_C_lvl = 0.0
TM_A_lvl = 0.0
TM_C_nrm = 0.0
TM_A_nrm = 0.0
for j in range(J):
    for i in range(M):
        for k in range(P):
            prob = ergodic_dist[j * MP + i * P + k]
            TM_C_lvl += cPol[j][i] * dist_pGrid[k] * prob
            TM_A_lvl += aPol[j][i] * dist_pGrid[k] * prob
            TM_C_nrm += cPol[j][i] * prob
            TM_A_nrm += aPol[j][i] * prob

print(f"TM Aggregate Consumption (level) = {TM_C_lvl:.6f}")
print(f"TM Aggregate Assets      (level) = {TM_A_lvl:.6f}")
print(f"TM Aggregate Consumption (norm)  = {TM_C_nrm:.6f}")
print(f"TM Aggregate Assets      (norm)  = {TM_A_nrm:.6f}")

## 6. Comparison

In [ ]:
print("=== Level Aggregates ===")
print(f"{'':20s} {'MC':>12s} {'TM':>12s} {'Diff':>12s}")
print(f"{'Consumption':20s} {MC_C:12.6f} {TM_C_lvl:12.6f} {MC_C - TM_C_lvl:12.6f}")
print(f"{'Assets':20s} {MC_A:12.6f} {TM_A_lvl:12.6f} {MC_A - TM_A_lvl:12.6f}")
print()
print("=== Normalized Aggregates ===")
print(f"{'':20s} {'MC':>12s} {'TM':>12s} {'Diff':>12s}")
print(
    f"{'Consumption':20s} {MC_C_nrm:12.6f} {TM_C_nrm:12.6f} {MC_C_nrm - TM_C_nrm:12.6f}"
)
print(f"{'Assets':20s} {MC_A_nrm:12.6f} {TM_A_nrm:12.6f} {MC_A_nrm - TM_A_nrm:12.6f}")

### Time series comparison

In [ ]:
dstn = ergodic_dist.copy()
tm_aLvls = []
for t in range(agent.T_sim - burn_in):
    A_val = 0.0
    for j in range(J):
        block = dstn[j * MP : (j + 1) * MP].reshape(M, P)
        for i in range(M):
            A_val += aPol[j][i] * np.dot(block[i, :], dist_pGrid)
    tm_aLvls.append(A_val)
    dstn = TranMatrix @ dstn

plt.figure(figsize=(16, 6))
plt.plot(mc_aLvls[burn_in:], label="Monte Carlo", alpha=0.7, linewidth=0.8)
plt.plot(tm_aLvls, label="Transition Matrix", linewidth=2.5)
plt.xlabel("Period")
plt.ylabel("Aggregate Assets (level)")
plt.title("MC vs TM: Serial Permanent Income Growth")
plt.legend(fontsize=12)
plt.show()

### Marginal distributions

Compare TM and MC marginal distributions over $m$ (integrating out $p$ and $j$)
and over $p$ (integrating out $m$ and $j$).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Marginal over m (aggregate across p and j)
tm_m_marginal = np.zeros(M)
for j in range(J):
    block = ergodic_dist[j * MP : (j + 1) * MP].reshape(M, P)
    tm_m_marginal += block.sum(axis=1)

axes[0].plot(dist_mGrid, tm_m_marginal, label="TM", linewidth=2)
h, _ = np.histogram(agent.state_now["mNrm"], bins=dist_mGrid)
h = h / h.sum()
axes[0].plot(dist_mGrid[:-1], h, label="MC", linewidth=1.5, alpha=0.8)
axes[0].set_xlabel("$m$ (normalized market resources)")
axes[0].set_ylabel("Probability")
axes[0].set_title("Marginal distribution of $m$")
axes[0].set_xlim([0, 15])
axes[0].legend()

# Marginal over p (aggregate across m and j)
tm_p_marginal = np.zeros(P)
for j in range(J):
    block = ergodic_dist[j * MP : (j + 1) * MP].reshape(M, P)
    tm_p_marginal += block.sum(axis=0)

axes[1].plot(dist_pGrid, tm_p_marginal, label="TM", linewidth=2)
h, _ = np.histogram(agent.state_now["pLvl"], bins=dist_pGrid)
h = h / h.sum()
axes[1].plot(dist_pGrid[:-1], h, label="MC", linewidth=1.5, alpha=0.8)
axes[1].set_xlabel("$p$ (permanent income level)")
axes[1].set_ylabel("Probability")
axes[1].set_title("Marginal distribution of $p$")
axes[1].legend()

plt.suptitle("Marginal Distributions: TM vs MC", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary

This notebook demonstrates the transition matrix method with a **2D grid** over
$(m, p)$ — the key extension needed when PermGroFac $\neq$ 1.0.

Key differences from the 1D prototypes:

1. **2D lottery:** `jump_to_grid_2D` distributes probability mass in the
   $(m, p)$ plane, preserving conditional means in both dimensions.

2. **Permanent income dynamics:** $p_{t+1} = p_t \cdot \psi \cdot \Gamma_{j'}$
   where $\Gamma_{j'}$ is the PermGroFac of the target Markov state.  This
   makes the $p$ distribution non-degenerate.

3. **Level vs normalized aggregates:** With heterogeneous $p$, aggregates in
   *level* terms ($C = \sum c_i p_i$) differ from normalized terms ($\bar{c} = \sum c_i$).
   Both are reported.

4. **Memory scaling:** The TM has $(M \times P \times J)^2$ entries.  With
   $M=80, P=25, J=5$: 10,000 states and ~800 MB.  For larger problems,
   **Harmenberg's neutral measure** collapses the $p$ dimension to get back to
   a 1D grid — that would be the natural next step.

### Next steps

- Implement Harmenberg's neutral measure to reduce back to a 1D grid
- Use sparse matrices for larger grids
- Tackle `AggShockMarkovConsumerType` (endogenous aggregate state)